## Running CL Simulations

We now have 
- Elo data
- A model that predicts number of goals based on Elo diff and whether or not there is a home advantage.

From this, we are able to simulate a Champions League tournament. To do so, we must research the current tournament format, and which teams are included. Note that we are focused on simulating Manchester United's run through the tournament primarily.

### League format

#### 1. Qualifying (we can ignore this as Manchester United have already qualified for the 26-27 tournament)

#### 2. League stage

There is one table containing all 36 teams. 

Each team plays 8 matches (4 home, 4 away) vs 8 different teams.

Teams are divided into 4 pots of 9, based on a UEFA club coefficient assignment system. Each club plays two teams from each pot, one home and one away. This is to prevent a club from facing a team from their own domestic league, and to stop a club from facing more than 2 clubs from any other domestic league. 

Manchester United's group phase would look like: 
- Pot 1: one home, one away 
- Pot 2: one home, one away 
- Pot 3: one home, one away 
- Pot 4: one home, one away 

The results are fed into one table as so:
- Win:     3 points 
- Draw:    1 point 
- Loss:   0 points

After 8 games:
- 1st–8th: directly to Round of 16
- 9th–24th: knockout play-off
- 25th–36th: eliminated

For teams tied in league rankings, the tiebreakers are goal (in order) goal difference, goals scored, away gaols scored, wins, away wins, various disciplinary criteria. We can choose a way to approximate deep ties (likely randomisation)

#### 3. Knockout play off

The teams finishing 9th - 24th play a 2 legged tie. The draw is:
- 9th / 10th vs 23rd / 24th  
- 11th / 12th vs 21st / 22nd  
- 13th / 14th vs 19th / 20th  
- 15th / 16th vs 17th / 18th

The higher ranked side plays at home in the second leg (irrelevant for our tournament).
The 8 winners join the top eight teams in the round of 16.

#### Round of 16 and onward

The top eight are seeded into the Ro16 based on their league position. 
All Ro16, quarter finals and semi finals matches are 2 legged aggegrate ties. 
The final is a single match.

Note (from UEFA):
'The new 2026/27 rules also preserve some league-phase seeding advantages further into the bracket: teams ranked 1–4 are due to play the second leg at home in the quarter-finals, and teams ranked 1–2 in the semi-finals. If they are knocked out, the team that beats them inherits that bracket position.'

## How our model will simulate this

#### 1. Qualifying

First gather the teams. 29 teams (including Man Utd) have already qualified:

Arsenal, Aston Villa, Athletico Madrid, Borussia Dortmund, Barcelona, Bayern Munich, Club Brugge, Como, Feyenoord, Galatasaray, Inter, leipzig, Lens, Lille, Liverpool, Man City, Man Utd, Napoli, Paris, Porto, PSV, Real Betis, Real Madrid, Roma, Shakhtar, Slavia Praha, Sporting CP, Stuttgart, Villareal

The remaining 7 places will be decided by playoffs in the coming weeks (18-26 Aug). Until then, we will randomise these playoffs to generate the list of teams. The playoffs are as follows:

Champions Path

- Levski Sofia        vs AEK Athens
- Celtic              vs LASK
- Dinamo Zagreb       vs Viking
- Slovan Bratislava   vs Celje
- Hapoel Be'er Sheva  vs Sabah

League Path

- Fenerbahçe          vs Lyon
- NEC                 vs Bodø/Glimt

Where no data exists in our model on the team, we can find data from elsewhere, or choose uniformly.

#### 2. League stage

We need to find out the exact rules behind the pots. 4 pots of 9, non random but assigned by UEFA's club coefficient rankings at the start of the season. The CL holders are placed as the top seed in Pot 1. So we will need to import UEFA coefficients into our data team (can be attached to the teams csv maybe)

After investigation of the possible qualifiers, (due to their low ranking) it would appear that Pots 1 and 2 are virtually fixed: \
Pot 1 \
PSG \
Bayern Munich \
Real Madrid \
Liverpool \
Inter \
Manchester City \
Arsenal \
Barcelona \
Atletico Madrid 

Pot 2 \
Borussia Dortmund \
Roma \
Sporting CP \
Aston Villa \
Porto\ 
Manchester United\ 
Club Brugge \
Real Betis \
PSV 

Pot 3 (contains 7 guaranteed teams:) \
Feyenoord \
Lille \
Napoli \
RB Leipzig \
Villarreal \
Shakhtar Donetsk \
Galatasaray, and 2 more places dependent on qualifying

Pot 4 (contains 4 guaranteed teams):
Slavia Prague \
Stuttgart \
Como \
Lens, with the remaining 5 places dependent on qualifying

It is possible to asssign each team to the correct pot from its coefficient. 

Once the pots are picked, we need a way to choose 2 teams from each pot for each team, such that none of the restrictions are breached. We coudl define a function that chooses a random pot choice, checks the tree choices for the tournament (check complexity), and rejects. Need to investigate how common a failure is. Can just run Monte carlo, and if failure comes ignore this trial.

Once teams from pots are chosen, simulating the league is realtively trivial.

#### 3. Knockout play-off

Then need to choose uniformly from the possible pairings and run the playoffs

#### 4. Round of 16 and onward

Seed top 8 into the top seeds
Seed next teams according to new 26/27 rules
Run sim





### Data and simulation prep

For convenience, we'll build a small dataframe containing 

team, team_id, country, elo, uefa coefficient, qualified. This will be easier to work with than our full database

In [56]:
import pandas as pd
import sqlite3
import soccerdata as sd
import numpy as np

In [48]:
connection = sqlite3.connect("../data/processed/football.db")

matches = pd.read_sql_query(
    "SELECT * FROM matches",
    connection
)

matches = matches.sort_values("date").reset_index(drop=True)

In [49]:
# calculate final Elo ratings at start of 2026/27 season

matches["date"] = pd.to_datetime(matches["date"])

elo_seasons = ["2015/16", "2016/17", "2017/18", "2018/19", "2019/20", "2020/21", "2021/22", "2022/23", "2023/24", "2024/25", "2025/26"]

matches = matches[matches["season"].isin(elo_seasons)].copy()
matches = matches.sort_values("date").reset_index(drop=True)


# prior elo values

clubelo = sd.ClubElo()
elo_2015 = clubelo.read_by_date("2015-06-30")

countries = ["ENG", "SCO", "GER", "ESP", "ITA", "FRA", "BEL", "NED", "POR", "TUR", "GRE"]

top_leagues = elo_2015[
    (elo_2015["country"].isin(countries)) &
    (elo_2015["level"] == 1)
]

league_elos = top_leagues.groupby("country")["elo"].mean()


league_to_country = {
    "E0": "ENG",
    "SC0": "SCO",
    "D1": "GER",
    "SP1": "ESP",
    "I1": "ITA",
    "F1": "FRA",
    "B1": "BEL",
    "N1": "NED",
    "P1": "POR",
    "T1": "TUR",
    "G1": "GRE"
}

team_leagues = pd.read_sql_query(
    "SELECT DISTINCT team_id, league FROM team_aliases WHERE source = 'football_data'",
    connection
)

final_elo = {}

for _, team in team_leagues.iterrows():

    league = team["league"]
    country = league_to_country[league]

    starting_rating = league_elos.loc[country]

    final_elo[team["team_id"]] = starting_rating


K = 20

for _, match in matches.iterrows():

    if match["home_team_id"] not in final_elo:
        final_elo[match["home_team_id"]] = 1500

    if match["away_team_id"] not in final_elo:
        final_elo[match["away_team_id"]] = 1500

    if match["home_goals"] > match["away_goals"]:
        S_A = 1
        S_B = 0

    elif match["home_goals"] == match["away_goals"]:
        S_A = 0.5
        S_B = 0.5

    else:
        S_A = 0
        S_B = 1

    current_home_rating = final_elo[match["home_team_id"]]
    current_away_rating = final_elo[match["away_team_id"]]

    Q_A = 10 ** (current_home_rating / 400)
    Q_B = 10 ** (current_away_rating / 400)

    expected_home_score = Q_A / (Q_A + Q_B)
    expected_away_score = Q_B / (Q_A + Q_B)

    updated_home_rating = current_home_rating + K * (S_A - expected_home_score)
    updated_away_rating = current_away_rating + K * (S_B - expected_away_score)

    final_elo[match["home_team_id"]] = updated_home_rating
    final_elo[match["away_team_id"]] = updated_away_rating


final_elo_df = pd.DataFrame(list(final_elo.items()), columns=["team_id", "elo"])

teams = pd.read_sql_query(
    "SELECT team_id, canonical_name, country FROM teams",
    connection
)

final_elo_df = final_elo_df.merge(
    teams,
    on="team_id",
    how="left"
)

final_elo_df = final_elo_df.sort_values("elo", ascending=False).reset_index(drop=True)

display(final_elo_df)

[08/14/26 12:22:32] INFO     Saving cached data to C:\Users\Rohan\soccerdata\data\ClubElo            _common.py:250

,team_id,elo,canonical_name,country
0,156,2050.360927,Bayern Munich,Germany
1,421,2018.523373,Barcelona,Spain
2,70,2008.559136,Arsenal,England
3,440,1989.862114,Real Madrid,Spain
4,140,1953.110627,Paris Saint-Germain,France
...,...,...,...,...
498,401,1210.976992,Partick Thistle,Scotland
499,403,1205.964895,Ross County,Scotland
500,404,1191.371834,St Johnstone,Scotland
501,394,1190.306453,Hamilton Academical,Scotland


In [50]:
# building dataset for simulation

qualified_teams = [70, 71, 420, 158, 421, 156, 31, 232, 301, 475, 240, 176, 127, 128, 85, 87, 88, 245, 140, 359, 310, 439, 440, 250, 499, 64, 363, 180, 447]

playoff_teams = [185, 390, 16, 55, 409, 413, 221, 474, 130, 308, 329]

uefa_coefficients = {70: 119.000, 71: 83.000, 420: 104.750, 158: 100.750, 421: 113.250, 156: 147.500, 31: 75.250, 232: 19.989, 301: 71.000, 475: 53.500, 240: 127.000, 176: 61.000, 127: 16.699, 128: 68.750, 85: 130.000, 87: 125.500, 88: 76.500, 245: 63.000, 140: 132.000, 359: 80.750, 310: 71.250, 439: 74.500, 440: 144.500, 250: 97.750, 499: 56.250, 64: 44.000, 363: 84.000, 180: 27.500, 447: 59.000, 185: 24.000, 390: 44.000, 16: 21.000, 55: 46.500, 409: 36.000, 413: 23.000, 221: 14.000, 474: 57.750, 130: 65.750, 308: 13.585, 329: 64.000}


tournament_teams_df = pd.read_sql_query(
    "SELECT team_id, canonical_name, country FROM teams WHERE team_id in (70, 71, 420, 158, 421, 156, 31, 232, 301, 475, 240, 176, 127, 128, 85, 87, 88, 245, 140, 359, 310, 439, 440, 250, 499, 64, 363, 180, 447, 185, 390, 16, 55, 409, 413, 221, 474, 130, 308, 329)",
    connection
)

tournament_teams_df["qualified"] = tournament_teams_df["team_id"].isin(qualified_teams).astype(int)

tournament_teams_df["elo"] = tournament_teams_df["team_id"].map(final_elo)
tournament_teams_df["uefa_coefficient"] = tournament_teams_df["team_id"].map(uefa_coefficients)

tournament_teams_df = tournament_teams_df[["team_id", "canonical_name", "country", "elo", "uefa_coefficient", "qualified"]]

tournament_teams_df = tournament_teams_df.sort_values(by="elo", ascending=False)

tournament_teams_df

,team_id,canonical_name,country,elo,uefa_coefficient,qualified
13,156,Bayern Munich,Germany,2050.360927,147.500,1
33,421,Barcelona,Spain,2018.523373,113.250,1
4,70,Arsenal,England,2008.559136,119.000,1
35,440,Real Madrid,Spain,1989.862114,144.500,1
12,140,Paris Saint-Germain,France,1953.110627,132.000,1
7,87,Manchester City,England,1933.987410,125.500,1
20,240,Inter Milan,Italy,1929.515784,127.000,1
14,158,Borussia Dortmund,Germany,1893.968345,100.750,1
32,420,Atletico Madrid,Spain,1878.984622,104.750,1
6,85,Liverpool,England,1862.694515,130.000,1


## Simulation

### 1. Qualifying and game simulation prep

The qualifying matchups are as follows:

- ~~Levski Sofia~~        vs AEK Athens
- Celtic              vs LASK
- Dinamo Zagreb       vs ~~Viking~~
- Slovan Bratislava   vs Celje
- Hapoel Be'er Sheva  vs ~~Sabah~~

League Path

- Fenerbahçe          vs Lyon
- NEC                 vs Bodø/Glimt

Our dataset does not include (Levski Sofia, Viking, or Sabah), so they will be auto DQ'd. The other games we will run simulations for.

It would be convenient to generate a function that takes two teams and returns a score based on our model.

In [52]:
# first we bring over our model data and build our model:

# building auxilliary table (key match_id) to matches, with per match elo data (elo before, elo after etc.)
# join to full table, to give full match data with elo 

elo = {}

for _, team in team_leagues.iterrows():

    league = team["league"]
    country = league_to_country[league]

    elo[team["team_id"]] = league_elos.loc[country]


home_elo_before = []
away_elo_before = []

home_elo_after = []
away_elo_after = []

expected_home_scores = []
expected_away_scores = []

K = 20

for _, match in matches.iterrows():

    if match["home_team_id"] not in elo:
        elo[match["home_team_id"]] = 1500

    if match["away_team_id"] not in elo:
        elo[match["away_team_id"]] = 1500

    current_home_rating = elo[match["home_team_id"]]
    current_away_rating = elo[match["away_team_id"]]

    home_elo_before.append(current_home_rating)
    away_elo_before.append(current_away_rating)

    Q_A = 10 ** (current_home_rating / 400)
    Q_B = 10 ** (current_away_rating / 400)

    expected_home_score = Q_A / (Q_A + Q_B)
    expected_away_score = Q_B / (Q_A + Q_B)

    expected_home_scores.append(expected_home_score)
    expected_away_scores.append(expected_away_score)

    if match["home_goals"] > match["away_goals"]:
        S_A = 1
        S_B = 0

    elif match["home_goals"] == match["away_goals"]:
        S_A = 0.5
        S_B = 0.5

    else:
        S_A = 0
        S_B = 1

    updated_home_rating = current_home_rating + K * (S_A - expected_home_score)
    updated_away_rating = current_away_rating + K * (S_B - expected_away_score)

    elo[match["home_team_id"]] = updated_home_rating
    elo[match["away_team_id"]] = updated_away_rating

    home_elo_after.append(updated_home_rating)
    away_elo_after.append(updated_away_rating)


elo_matches = matches.copy()

elo_matches["home_elo_before"] = home_elo_before
elo_matches["away_elo_before"] = away_elo_before

elo_matches["home_elo_after"] = home_elo_after
elo_matches["away_elo_after"] = away_elo_after

elo_matches["expected_home_score"] = expected_home_scores
elo_matches["expected_away_score"] = expected_away_scores

elo_matches["elo_diff"] = (
    elo_matches["home_elo_before"] - elo_matches["away_elo_before"]
)

# adding line by line input data (need 2 rows per match H and A)
home_elo_matches = elo_matches.copy()
home_elo_matches['elo_diff'] = home_elo_matches['home_elo_before'] - home_elo_matches['away_elo_before']
home_elo_matches['H'] = 1
home_elo_matches['goals'] = home_elo_matches['home_goals']

away_elo_matches = elo_matches.copy()
away_elo_matches['elo_diff'] = away_elo_matches['away_elo_before'] - away_elo_matches['home_elo_before']
away_elo_matches['H'] = 0
away_elo_matches['goals'] = away_elo_matches['away_goals']

doubled_elo_matches = pd.concat([home_elo_matches, away_elo_matches])

doubled_elo_matches = doubled_elo_matches.sort_values(
    ['date', 'match_id']
)

# filtering data to what we need

ml_table = doubled_elo_matches.copy()

ml_table = ml_table[
    [
        "match_id",
        "date",
        "season",
        "competition",
        "home_team_name",
        "away_team_name",
        "elo_diff",
        "H",
        "goals"
    ]
]

ml_table = ml_table[
    ml_table["season"].isin([
        "2017/18",
        "2018/19",
        "2019/20",
        "2020/21",
        "2021/22",
        "2022/23",
        "2023/24",
        "2024/25",
        "2025/26"
    ])
]

ml_table = ml_table.sort_values(["date", "match_id", "H"], ascending=[True, True, False]).reset_index(drop=True)





In [53]:

# final model using all historical data

import statsmodels.api as sm

full_train = ml_table.copy()

X = full_train[["elo_diff", "H"]]
y = full_train["goals"]

X = sm.add_constant(X)

forecast_model = sm.GLM(
    y,
    X,
    family=sm.families.Poisson()
)

forecast_results = forecast_model.fit()

forecast_results.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                  goals   No. Observations:                66756
Model:                            GLM   Df Residuals:                    66753
Model Family:                 Poisson   Df Model:                            2
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -97662.
Date:                Fri, 14 Aug 2026   Deviance:                       75498.
Time:                        12:22:37   Pearson chi2:                 6.60e+04
No. Iterations:                     5   Pseudo R-squ. (CS):             0.1349
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1691      0.005     33.732      0.000       0.159       0.179
elo_diff       0.0020   2.17e-05     92.730      0.000       0.002       0.002
H              0.2247      0.007     34.078      0.000       0.212       0.238
==============================================================================
"""

In [93]:
def simulate_match(hometeam, awayteam):
    # generate poisson variable for home and away (need elo diff, and home or not home)
    
    home_elo = tournament_teams_df.loc[tournament_teams_df["team_id"] == hometeam, "elo"].iloc[0]
    away_elo = tournament_teams_df.loc[tournament_teams_df["team_id"] == awayteam, "elo"].iloc[0]

    home_elo_diff = home_elo - away_elo
    away_elo_diff = away_elo - home_elo

    # home goals:
    home_match = pd.DataFrame({
        "const": [1],
        "elo_diff": [home_elo_diff],
        "H": [1] 
        
    })

    # away goals:
    away_match = pd.DataFrame(
        {
        "const": [1],
        "elo_diff": [away_elo_diff],
        "H": [0] 
        }
    )

    home_lambda = forecast_results.predict(home_match)
    away_lambda = forecast_results.predict(away_match)

    home_goals = np.random.poisson(home_lambda)
    away_goals = np.random.poisson(away_lambda)

    return home_goals, away_goals

# extra time

def simulate_extra_time(hometeam, awayteam):

    home_elo = tournament_teams_df.loc[tournament_teams_df["team_id"] == hometeam, "elo"].iloc[0]
    away_elo = tournament_teams_df.loc[tournament_teams_df["team_id"] == awayteam, "elo"].iloc[0]

    home_elo_diff = home_elo - away_elo
    away_elo_diff = away_elo - home_elo

    home_match = pd.DataFrame({
        "const": [1],
        "elo_diff": [home_elo_diff],
        "H": [1]
    })

    away_match = pd.DataFrame({
        "const": [1],
        "elo_diff": [away_elo_diff],
        "H": [0]
    })

    home_lambda = forecast_results.predict(home_match).iloc[0]
    away_lambda = forecast_results.predict(away_match).iloc[0]

    # extra time is 30 minutes rather than 90
    home_goals = np.random.poisson(home_lambda / 3)
    away_goals = np.random.poisson(away_lambda / 3)

    return home_goals, away_goals

In [ ]:
# test run
home_team = 156 # bayern munich at home
away_team = 127 # lens away

home_wins = 0
draws = 0
away_wins = 0

for i in range(20):

    home_goals, away_goals = simulate_match(home_team, away_team)

    print(home_goals, "-", away_goals)

    if home_goals > away_goals:
        home_wins += 1

    elif home_goals == away_goals:
        draws += 1

    else:
        away_wins += 1

print()
print("home wins:", home_wins)
print("draws:", draws)
print("away wins:", away_wins)

[6] - [1]
[6] - [2]
[3] - [1]
[3] - [5]
[2] - [2]
[3] - [0]
[0] - [0]
[4] - [0]
[2] - [2]
[5] - [2]
[1] - [0]
[1] - [0]
[3] - [1]
[7] - [1]
[4] - [0]
[5] - [0]
[1] - [1]
[2] - [0]
[3] - [1]
[3] - [0]

Home wins: 15
Draws: 4
Away wins: 1


In [92]:
home_team = 140 # psg at home]
away_team = 250 # roma away

home_wins = 0
draws = 0
away_wins = 0

for i in range(20):

    home_goals, away_goals = simulate_match(home_team, away_team)

    print(home_goals, "-", away_goals)

    if home_goals > away_goals:
        home_wins += 1

    elif home_goals == away_goals:
        draws += 1

    else:
        away_wins += 1

print()
print("home wins:", home_wins)
print("draws:", draws)
print("away wins:", away_wins)

[0] - [1]
[0] - [0]
[2] - [0]
[3] - [2]
[3] - [0]
[4] - [1]
[1] - [0]
[2] - [2]
[3] - [0]
[2] - [0]
[3] - [2]
[2] - [2]
[3] - [1]
[2] - [2]
[1] - [0]
[4] - [1]
[2] - [0]
[1] - [0]
[2] - [0]
[0] - [1]

home wins: 14
draws: 4
away wins: 2


#### Comments:

We can see that this simulation works, but this exposes a problem in the model. Lambda increases continuously with elo_diff, and has no "checks" on predicting goals. ie if there is a 700 elo gap, it'll just expect the dominant team to score 5 or 6 goals, whilst in reality they are more likely to score 2 or 3 and cruise to the end to save energy. This is not a major issue given that we're only interested in W/L results here, but may slightly exaggerate win probabilities for huge mismatches.

This could be fixed by considering elo "bands" and their difference? eg How different is a gap of 300 and 400 elo on average? There will be a limit to the point where elo diff makes a difference to the number of goals scored. 

#### Next steps

We can now use the simulation function to simulate our qualifying round. We will build the rounds as functions, outputting sufficient data for the next round. 

Need to decide how to settle - extra time and penalties. Extra time can be simulated as a poisson still , with 1/3 of the expected goals (and therefore 1/3 of the lambda). penalties can be settled randomly.

In [94]:
# qualifying round

def simulate_qualifying():

    qualifying_df = tournament_teams_df.copy()

    # automatically qualified

    for auto_qualify in [55, 185, 221]:
        qualifying_df.loc[qualifying_df["team_id"] == auto_qualify, "qualified"] = 1


    # celtic vs lask

    leg_1 = simulate_match(390, 16)
    leg_2 = simulate_match(16, 390)

    celtic_goals = leg_1[0] + leg_2[1]
    lask_goals = leg_1[1] + leg_2[0]

    if celtic_goals > lask_goals:
        winner = 390

    elif lask_goals > celtic_goals:
        winner = 16

    else:

        extra_time = simulate_extra_time(16, 390)

        lask_extra_goals = extra_time[0]
        celtic_extra_goals = extra_time[1]

        if celtic_extra_goals > lask_extra_goals:
            winner = 390

        elif lask_extra_goals > celtic_extra_goals:
            winner = 16

        else:
            winner = np.random.choice([390, 16])

    qualifying_df.loc[qualifying_df["team_id"] == winner, "qualified"] = 1


    # slovan bratislava vs celje

    leg_1 = simulate_match(409, 413)
    leg_2 = simulate_match(413, 409)

    slovan_goals = leg_1[0] + leg_2[1]
    celje_goals = leg_1[1] + leg_2[0]

    if slovan_goals > celje_goals:
        winner = 409

    elif celje_goals > slovan_goals:
        winner = 413

    else:

        extra_time = simulate_extra_time(413, 409)

        celje_extra_goals = extra_time[0]
        slovan_extra_goals = extra_time[1]

        if slovan_extra_goals > celje_extra_goals:
            winner = 409

        elif celje_extra_goals > slovan_extra_goals:
            winner = 413

        else:
            winner = np.random.choice([409, 413])

    qualifying_df.loc[qualifying_df["team_id"] == winner, "qualified"] = 1


    # fenerbahce vs lyon

    leg_1 = simulate_match(474, 130)
    leg_2 = simulate_match(130, 474)

    fenerbahce_goals = leg_1[0] + leg_2[1]
    lyon_goals = leg_1[1] + leg_2[0]

    if fenerbahce_goals > lyon_goals:
        winner = 474

    elif lyon_goals > fenerbahce_goals:
        winner = 130

    else:

        extra_time = simulate_extra_time(130, 474)

        lyon_extra_goals = extra_time[0]
        fenerbahce_extra_goals = extra_time[1]

        if fenerbahce_extra_goals > lyon_extra_goals:
            winner = 474

        elif lyon_extra_goals > fenerbahce_extra_goals:
            winner = 130

        else:
            winner = np.random.choice([474, 130])

    qualifying_df.loc[qualifying_df["team_id"] == winner, "qualified"] = 1


    # nec vs bodo/glimt

    leg_1 = simulate_match(308, 329)
    leg_2 = simulate_match(329, 308)

    nec_goals = leg_1[0] + leg_2[1]
    bodo_goals = leg_1[1] + leg_2[0]

    if nec_goals > bodo_goals:
        winner = 308

    elif bodo_goals > nec_goals:
        winner = 329

    else:

        extra_time = simulate_extra_time(329, 308)

        bodo_extra_goals = extra_time[0]
        nec_extra_goals = extra_time[1]

        if nec_extra_goals > bodo_extra_goals:
            winner = 308

        elif bodo_extra_goals > nec_extra_goals:
            winner = 329

        else:
            winner = np.random.choice([308, 329])

    qualifying_df.loc[qualifying_df["team_id"] == winner, "qualified"] = 1


    # keep only teams entering the league stage

    qualified_df = qualifying_df[
        qualifying_df["qualified"] == 1
    ].copy()

    return qualified_df

### League Phase

The pots are built in order of coefficients, so we can build a simple pot column and add to qualified_df. We can then fill the pots easily in another column.

We can then separate the draw into blocks of pot1 vs pot2 fixtures, pot2 v pot1, etc. (each pot has its own same-pot block). This means it's always possible for each team to get 2 opponenents from each pot, with 4  home and 4 away games.

Then we build each block, randomly pairing legal opponent pairings. If it reaches a team that cannot be paired, we return to an earlier pairing, and retry. Conditions for a legal match are (not already played, from different countries, doesn't exceed the max of two opps from other countries)

Then we generate the league draw, and run the simulations. Fill the league table, and proceed to the next round. 

In [ ]:
# assign the 4 pots

def assign_pots(league_teams_df, holder_id=140): # 140 is psg

    league_df = league_teams_df.copy()

    
    if len(league_df) != 36:
        raise ValueError("league phase must contain 36 teams")

    if league_df["team_id"].nunique() != 36:
        raise ValueError("duplicate team ids found")

    if league_df["uefa_coefficient"].isna().any():
        raise ValueError("missing uefa coefficient")

    if holder_id not in league_df["team_id"].values:
        raise ValueError("holder not found")


    holder = league_df[
        league_df["team_id"] == holder_id
    ].copy()

    other_teams = league_df[
        league_df["team_id"] != holder_id
    ].copy()

    other_teams = other_teams.sort_values(
        "uefa_coefficient",
        ascending=False
    )

    league_df = pd.concat([
        holder,
        other_teams
    ]).reset_index(drop=True)


    league_df["pot"] = 0

    league_df.loc[0:8, "pot"] = 1
    league_df.loc[9:17, "pot"] = 2
    league_df.loc[18:26, "pot"] = 3
    league_df.loc[27:35, "pot"] = 4

    return league_df

In [ ]:
# error checking function
def check_feasibility(league_teams_df):

    problems = []

    if len(league_teams_df) != 36:
        problems.append("field does not contain 36 teams")

    if league_teams_df["team_id"].nunique() != 36:
        problems.append("field contains duplicate team ids")


    # check each pot

    for pot in [1, 2, 3, 4]:

        pot_df = league_teams_df[
            league_teams_df["pot"] == pot
        ]

        if len(pot_df) != 9:
            problems.append("pot " + str(pot) + " does not contain 9 teams")

        country_counts = pot_df["country"].value_counts()

        for country, count in country_counts.items():

            if count > 4:
                problems.append(
                    "too many teams from " + country +
                    " in pot " + str(pot)
                )


    # check pairs of pots

    for pot_1 in [1, 2, 3, 4]:

        for pot_2 in [pot_1 + 1, pot_1 + 2, pot_1 + 3]:

            if pot_2 > 4:
                continue

            pot_1_df = league_teams_df[
                league_teams_df["pot"] == pot_1
            ]

            pot_2_df = league_teams_df[
                league_teams_df["pot"] == pot_2
            ]

            countries = set(pot_1_df["country"]) | set(pot_2_df["country"])

            for country in countries:

                number_in_pot_1 = sum(pot_1_df["country"] == country)
                number_in_pot_2 = sum(pot_2_df["country"] == country)

                if number_in_pot_1 + number_in_pot_2 > 9:
                    problems.append(
                        country + " makes pots " +
                        str(pot_1) + " and " +
                        str(pot_2) + " impossible"
                    )

    return problems

In [96]:
# check if a proposed fixture is allowed (to be used during draw, needs to track opponents and country_counts. since only 2 opponents from a single country is allowed)

def is_valid_match(
    home_team,
    away_team,
    team_country,
    opponents,
    country_counts
):

    # cannot play yourself

    if home_team == away_team:
        return False


    # cannot play the same opponent twice

    if away_team in opponents[home_team]:
        return False


    home_country = team_country[home_team]
    away_country = team_country[away_team]


    # cannot play a team from your own country

    if home_country == away_country:
        return False


    # cannot play more than two teams from another country

    home_country_count = country_counts[home_team].get(
        away_country,
        0
    )

    away_country_count = country_counts[away_team].get(
        home_country,
        0
    )

    if home_country_count >= 2:
        return False

    if away_country_count >= 2:
        return False


    return True

In [ ]:
# build draw "block" . eg say block is pot 1 home vs pot 2 away, we need 9 home and 9 away teams.
# if we hit an invalid node, we undo the previous team choice and try again

def find_matching(
    home_teams,
    away_teams,
    team_country,
    opponents,
    country_counts,
    fixtures,
    rng
):

    home_teams = home_teams.copy()
    away_teams = away_teams.copy()

    rng.shuffle(home_teams)


    def try_next_team(remaining_home, remaining_away):

        # everyone has been matched

        if len(remaining_home) == 0:
            return True


        home_team = remaining_home[0]

        possible_away_teams = remaining_away.copy()

        rng.shuffle(possible_away_teams)


        for away_team in possible_away_teams:

            valid = is_valid_match(
                home_team,
                away_team,
                team_country,
                opponents,
                country_counts
            )

            if valid:

                home_country = team_country[home_team]
                away_country = team_country[away_team]


                # add the match

                fixtures.append({
                    "home_team_id": home_team,
                    "away_team_id": away_team
                })

                opponents[home_team].add(away_team)
                opponents[away_team].add(home_team)

                country_counts[home_team][away_country] = (
                    country_counts[home_team].get(away_country, 0) + 1
                )

                country_counts[away_team][home_country] = (
                    country_counts[away_team].get(home_country, 0) + 1
                )


                # try to match the remaining teams

                next_home = remaining_home[1:]

                next_away = [
                    team for team in remaining_away
                    if team != away_team
                ]

                worked = try_next_team(
                    next_home,
                    next_away
                )

                if worked:
                    return True


                # if it failed, undo the match

                fixtures.pop()

                opponents[home_team].remove(away_team)
                opponents[away_team].remove(home_team)

                country_counts[home_team][away_country] -= 1
                country_counts[away_team][home_country] -= 1

                if country_counts[home_team][away_country] == 0:
                    del country_counts[home_team][away_country]

                if country_counts[away_team][home_country] == 0:
                    del country_counts[away_team][home_country]


        # no possible opponent worked

        return False


    return try_next_team(
        home_teams,
        away_teams
    )

In [ ]:
# assume we have a draw. we check now that it is valid

def validate_draw(league_teams_df, fixtures_df):

    problems = []


    if len(league_teams_df) != 36:
        problems.append("field does not contain 36 teams")

    if league_teams_df["team_id"].nunique() != 36:
        problems.append("field contains duplicate team ids")

    if len(fixtures_df) != 144:
        problems.append("draw does not contain 144 fixtures")


    team_country = dict(zip(
        league_teams_df["team_id"],
        league_teams_df["country"]
    ))

    team_pot = dict(zip(
        league_teams_df["team_id"],
        league_teams_df["pot"]
    ))

    team_ids = league_teams_df["team_id"].tolist()


    home_games = {
        team_id: 0
        for team_id in team_ids
    }

    away_games = {
        team_id: 0
        for team_id in team_ids
    }

    opponents = {
        team_id: []
        for team_id in team_ids
    }


    for _, match in fixtures_df.iterrows():

        home_team = match["home_team_id"]
        away_team = match["away_team_id"]


        if home_team not in team_ids or away_team not in team_ids:
            problems.append("fixture contains unknown team")
            continue


        if home_team == away_team:
            problems.append("team drawn against itself")


        if team_country[home_team] == team_country[away_team]:
            problems.append("same-country fixture found")


        home_games[home_team] += 1
        away_games[away_team] += 1

        opponents[home_team].append(away_team)
        opponents[away_team].append(home_team)


    # check each team's draw

    for team_id in team_ids:

        if home_games[team_id] != 4:
            problems.append(
                str(team_id) + " does not have 4 home games"
            )

        if away_games[team_id] != 4:
            problems.append(
                str(team_id) + " does not have 4 away games"
            )


        if len(opponents[team_id]) != 8:
            problems.append(
                str(team_id) + " does not have 8 opponents"
            )


        if len(set(opponents[team_id])) != 8:
            problems.append(
                str(team_id) + " has duplicate opponents"
            )


        # exactly two opponents from each pot

        for pot in [1, 2, 3, 4]:

            number_from_pot = 0

            for opponent in opponents[team_id]:

                if team_pot[opponent] == pot:
                    number_from_pot += 1

            if number_from_pot != 2:
                problems.append(
                    str(team_id) +
                    " does not have 2 opponents from pot " +
                    str(pot)
                )


        # maximum two opponents from another country

        opponent_country_counts = {}

        for opponent in opponents[team_id]:

            country = team_country[opponent]

            opponent_country_counts[country] = (
                opponent_country_counts.get(country, 0) + 1
            )


        for country, count in opponent_country_counts.items():

            if count > 2:
                problems.append(
                    str(team_id) +
                    " has more than 2 opponents from " +
                    country
                )


    return problems